# Lab 4 - Divide and Conquer Algorithms

## Problem 1: Selection Problem

### Problem Statement

The selection problem asks us to find the kth smallest element in a sequence of n numbers, where n >= 2 and 1 <= k <= n.

Input: A sequence of n numbers and an integer k.

Output: The kth smallest number in the sequence.

For example, if we are given the list:
[5, 1, 6, 7, 3, 4, 8]
and asked to find k = 3, then the answer is 4 because the sorted order is:
[1, 3, 4, 5, 6, 7, 8]
and the 3rd smallest value is 4.

This problem is commonly used in statistics when trying to find a median or a percentile of a dataset.

### Approach

The simplest solution is to sort the array and return the element at index k - 1. However, this takes O(n log n) time using a comparison-based sort.

A more efficient divide-and-conquer approach is to use a randomized quickselect algorithm:
- Pick a random pivot value from the list.
- Partition the list so that values smaller than the pivot go to the left and values larger than the pivot go to the right.
- The pivot ends up in its final sorted position.
- If that position is equal to k - 1, then the pivot is the answer.
- If the target index is smaller, repeat on the left partition.
- If the target index is larger, repeat on the right partition.

This works because each partition step removes one part of the list from consideration, and the pivot’s final position tells us how many elements are smaller than it.

### Illustration 1

Example list:
[7, 2, 9, 1, 5, 8, 3]

Suppose we want the 4th smallest element, so k = 4.

Choose pivot = 5.

Partition around 5:
[2, 1, 3, 5, 7, 9, 8]

The pivot 5 is now at index 3 (0-based), which means it is the 4th smallest element.
Since k = 4, the answer is 5.

### Illustration 2

Example list:
[5, 1, 6, 7, 3, 4, 8]

Suppose we want the 3rd smallest element, so k = 3.

Choose pivot = 6.

Partition around 6:
[5, 1, 3, 4, 6, 7, 8]

The pivot 6 is now at index 4 (0-based), which means it is the 5th smallest element.
Because we are looking for the 3rd smallest element, we continue searching the left partition:
[5, 1, 3, 4]

The algorithm repeats until the target rank is found.

### High-Level Pseudocode

```text
SelectionProblem(A, k)

    if length(A) == 1
        return A[0]

    pivot ← chooseRandomPivot(A)
    left, pivotValue, right ← partition(A, pivot)

    j ← index of pivotValue in the partitioned array

    if k - 1 == j
        return pivotValue
    else if k - 1 < j
        return SelectionProblem(left, k)
    else
        return SelectionProblem(right, k - j - 1)
```

### Correctness Justification

The algorithm is correct because it uses the same logic as the partition step in quicksort. After choosing a pivot and partitioning the list, all elements to the left of the pivot are smaller than or equal to the pivot, and all elements to the right are larger than or equal to the pivot. This ensures the pivot is placed in its true sorted position, so its index tells us the exact rank of the pivot in the final ordering.

If the pivot’s index matches the desired rank k - 1, then the pivot is exactly the kth smallest element and the algorithm returns it. If the desired rank is smaller than the pivot’s index, then the kth smallest element must be in the left partition, and the algorithm recursively searches there. If the desired rank is larger, then the kth smallest element must be in the right partition, and the algorithm recursively searches there with the correct adjusted rank. Because the algorithm only discards partitions that cannot contain the answer, it eventually narrows the problem to a smaller subarray that contains the kth smallest element. Therefore, the algorithm always returns the correct result.

### Implementation and Asymptotic Runtime Analysis

*Completed by @BradenEverson*

In [4]:
import random

def choose_random_pivot(A):
    return random.randrange(len(A))

def partition(A, pivot):
    pivot_value = A[pivot]
    left, right = [], []
    for i, x in enumerate(A):
        if i == pivot:
            continue
        if x < pivot_value:
            left.append(x)
        else:
            right.append(x)
    return left, pivot_value, right


def selection_problem(A, k):
    if len(A) == 1:
        return A[0]

    pivot = choose_random_pivot(A)
    left, pivot_value, right = partition(A, pivot)
    j = len(left)

    if k - 1 == j:
        return pivot_value
    elif k - 1 < j:
        return selection_problem(left, k)
    else:
        return selection_problem(right, k - j - 1)

### Asymptotic Runtime Analysis

`choose_random_pivot` is a constant function in $\Theta$(1), partition makes a single pass over all elements so it is in $\Theta$(n). `selection_problem` itself must be examined in its different cases based on how the pivot functions.

In the best case, the pivot is the k-th smallest element and the function returns immediately after one partition, leading to $\Theta(n)$

In the worst case, the pivot is the absolute worst and selects the greatest or smallest element each time and leads to the recurrence of $T(n) = T(n-1) + n$ or $\Theta(n^2)$

In the average case, the pivot is in the middle of the list and resolves to the recurrence $T(n) = T(3n/4) + n$ or $\Theta(n)$

### Test Cases

*Completed by @BradenEverson*

In [10]:
import random
import pandas as pd

small = [7, 2, 9, 4, 1]
sorted_arr = [1, 2, 3, 4, 5, 6, 7, 8]
reversed_arr = [8, 7, 6, 5, 4, 3, 2, 1]
all_same = [4, 4, 4, 4, 4, 4]
negatives = [-3, 10, -7, 0, 2, -1]
single = [6]
pair = [9, 3]
big = list(range(10_000))
random.Random(0).shuffle(big)

tests = [
    ("Small, minimum (k=1)", small, 1, 1),
    ("Small, maximum (k=n)", small, 5, 9),
    ("Small, median", small, 3, 4),
    ("Already sorted, middle", sorted_arr, 4, 4),
    ("Reverse sorted, k=2", reversed_arr, 2, 2),
    ("Duplicates", dupes, 2, 3),
    ("All values equal", all_same, 4, 4),
    ("Single element (base case)", single, 1, 6),
    ("Two elements, k=1", pair, 1, 3),
    ("Two elements, k=2", pair, 2, 9),
    ("Large (n=10,000), median", big, 5_000, 4_999),
]

rows = []
for name, arr, k, expected in tests:
    actual = selection_problem(arr, k)
    rows.append([name, len(arr), k, expected, actual, expected == actual])

pd.DataFrame(rows, columns=["Test", "n", "k", "Expected", "Actual", "Pass"])

,Test,n,k,Expected,Actual,Pass
0,"Small, minimum (k=1)",5,1,1,1,True
1,"Small, maximum (k=n)",5,5,9,9,True
2,"Small, median",5,3,4,4,True
3,"Already sorted, middle",8,4,4,4,True
4,"Reverse sorted, k=2",8,2,2,2,True
5,Duplicates,6,2,3,3,True
6,All values equal,6,4,4,4,True
7,Single element (base case),1,1,6,6,True
8,"Two elements, k=1",2,1,3,3,True
9,"Two elements, k=2",2,2,9,9,True


### Benchmark Results

*Completed by @BrettStoroe*

In [ ]:
# Benchmarking Code
# Completed by Team Member #3

### Benchmark Results

*Completed by @BrettStoroe*

### Team Member Contributions

#### Kaiden Pollesch
- Problem approach description
- Illustrations
- High-level pseudocode
- Correctness justification

#### Braden Everson
- Algorithm implementation
- Runtime analysis
- Test cases

#### Brett Storoe
- Benchmarking
- Benchmarking results
